---
numbering: false
---

# GEE ERA5 data

Extract a monthly time-series of [ERA5-Land](https://developers.google.com/earth-engine/datasets/catalog/ECMWF_ERA5_LAND_MONTHLY_AGGR) climate reanalysis variables at a point using Google Earth Engine (GEE), and save it as a CSV file.

In [ ]:
import ee  # The main Earth Engine library

import pandas as pd

import matplotlib.pyplot as plt

Connect to Google Earth Engine. You will need to provide your own project ID (if in doubt, go to [code.earthengine.google.com](https://code.earthengine.google.com/) and see the name in the top-right corner).

In [ ]:
# =====================================
# Replace this with your own project id
my_project_name = "ee-trchudley"
# =====================================

ee.Authenticate()
ee.Initialize(project=my_project_name)

## Set parameters

Set the location, date range and variables. A complete catalogue of variables, including units and descriptions, can be found in the ['bands' tab of the Earth Engine documentation](https://developers.google.com/earth-engine/datasets/catalog/ECMWF_ERA5_LAND_MONTHLY_AGGR#bands).

In [ ]:
# Location - editable
latitude = 41.017          # Degrees of latitude
longitude = -123.611       # Degrees of longitude
location_name = "hoopa"    # recognisable name, to create a useful file name

# Date range in the format "YYYY-MM-DD" (the end date is exclusive)
date_min, date_max = "2013-01-01", "2026-01-01"

# Names of the variables we want to download -- add more if you like!
variables = [
    "temperature_2m",
    "total_precipitation_sum",
]

# Create a point geometry at the specified location
point = ee.Geometry.Point(longitude, latitude)

## Extract ERA5 data

Filter the ERA5-Land monthly collection to our dates and variables, and extract the value of each variable at the pixel that overlaps our point. These are stored as an `era5` property of each image.

In [ ]:
# Define the dataset as the ERA5-Land monthly aggregated data
dataset = ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR")

# Filter the dataset to the desired time range, and select only the relevant variables
dataset = dataset.filterDate(date_min, date_max).select(variables)

# Go through the collection, finding the values at the pixel overlapping our point
dataset = dataset.map(
    lambda image: image.set(
        "era5",
        image.reduceRegion(
            reducer=ee.Reducer.first(),
            geometry=point,
            scale=image.projection().nominalScale()  # Use native resolution
        ).set("year", image.date().get("year")).set("month", image.date().get("month"))
    )
)

Download the data and convert to a `pandas` dataframe:

In [ ]:
# Get the data as a list of dictionaries, and turn it into a pandas dataframe
era5 = pd.DataFrame(dataset.aggregate_array("era5").getInfo())

# Construct a 'date' column from the year and month data, with a nominal date of the 15th of the month
dates = pd.to_datetime(dict(year=era5.year, month=era5.month, day=15))

# Replace the 'year' and 'month' columns with the 'date' column, and order columns as requested
era5 = era5.drop(columns=["year", "month"])
era5 = era5[variables]
era5.insert(0, "date", dates)

era5

Convert units to be human-readable: temperature from Kelvin to Celsius, and precipitation from metres to millimetres. If you've included any additional variables, consult the documentation and add corrections of your own here.

In [ ]:
# Create new 'corrected' table as a copy of the first
era5_corr = era5.copy()

# Convert temperature from Kelvin to Celsius
if "temperature_2m" in era5_corr:
    era5_corr["temperature_2m"] = era5_corr["temperature_2m"] - 273.15

# Convert total monthly precipitation from m to mm
if "total_precipitation_sum" in era5_corr:
    era5_corr["total_precipitation_sum"] = era5_corr["total_precipitation_sum"] * 1000

era5_corr

Plot each variable:

In [ ]:
fig, axes = plt.subplots(nrows=len(variables), sharex=True, layout="constrained", figsize=(8, 2.5 * len(variables)), squeeze=False)

for ax, variable in zip(axes.flat, variables):
    era5_corr.plot("date", variable, ax=ax, legend=False)
    ax.set_ylabel(variable)
    ax.grid()

ax.set_xlabel("Date")

plt.show()

Save this under a sensible name, auto-generated from parameters:

In [ ]:
filename = f"{location_name}_era5_monthly_{date_min}_{date_max}.csv"

print("Saving to:", filename)

era5_corr.to_csv(filename, index=False)